In [ ]:
# %pip install git+https://github.com/Mishne-Lab/pyRATS
# if not already installed on the system:
# %pip install pandas imageio seaborn matplotlib

from pyRATS import rats
from examples import vis

# Load data

In [ ]:
import os
import pickle

def path_exists(path):
    return os.path.exists(path) or os.path.islink(path)

def read(fpath, verbose=True):
    if not path_exists(fpath):
        if verbose:
            print(fpath, 'does not exist.')
        return None
    with open(fpath, "rb") as f:
        data = pickle.load(f)
    if verbose:
        print('Read data from', fpath, flush=True)
    return data
    

In [ ]:
data_fname = '../data/data.dat'         # PATH TO DATA
X, labels, metadata = read(data_fname)


# Generate embeddings

In [ ]:
model = rats.RATS(
    n_components=2, 
    kernel='cosine', 
    n_neighbors=34, 
    cost_function='distortion', 
    min_cluster_size=3, 
    verbose=True, 
    metric='cosine',
    n_iter_without_progress=20,
    tree='mst',
    root_view= 'largest',
    tear=True,
)
y = model.fit_transform(X)

In [ ]:
tear_color_eig_inds = [5, 1, 3] # interior has green color so assigned smallest value of i to g-channel
color_of_pts_on_tear = model.compute_color_of_pts_on_tear(
    y,
    tear_color_eig_inds=tear_color_eig_inds,
)
vis.Visualize().global_embedding(
    y, labels[:,0],
    color_of_pts_on_tear=color_of_pts_on_tear[:,tear_color_eig_inds] if color_of_pts_on_tear is not None else None,
    cmap0='summer',
    cmap1='jet',
    title='color='+str(tear_color_eig_inds),
    figsize=(3,3)
)

To visuallly smoothen the edges, we save the data and use the cut-and-paste app. 
To run the app, execute:
- bokeh serve cut_and_paste_app.py --allow-websocket-origin=localhost:5006
- then go to http://localhost:5006

In [ ]:
def save(dirpath, fname, data, verbose=True):
    if not path_exists(dirpath):
        os.makedirs(dirpath)
    fpath = dirpath + '/' + fname
    with open(fpath, "wb") as f:
        pickle.dump(data, f)
    if verbose:
        print('Saved data in', fpath, flush=True)
        
emb_info = {
    'y': y,
    'color_of_pts_on_tear': color_of_pts_on_tear,
    'model': model,
}
metadata = {
    'algo': 'rats',
    'hyperparameters': {
        'd': 2,
    },
}
save('./generated_data/rats/34_3/', 'rats.dat', [emb_info, metadata])